**Purpose:** LLM-assisted keyword augmentation for the sector news queries.

**Inputs:** `gics_word_labeling_3_labeled.parquet` (one-off manually labeled file, not in repo), `news/gics_description.xlsx`, `news/gics_word_labeling_0.parquet`, `news/gics_word_labeling_1.parquet`, `news/gics_word_labeling_2.parquet`, `news/gics_word_labeling_full.parquet`, `news/subind_words-2.parquet`, `news/subind_words-3.parquet`

**Outputs:** `gics_word_labeling.parquet`, `news/gics_word_labeling_full.parquet`, `news/subind_words_full.parquet`, `subind_words.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


# LLMs

instruct beats normal (for LLMs choice)

all llms selected were instruct, around 7B parameters in order to be comparable and run locally, open-sourced, and metioned in the related literature

#### review this:

The purpose of this prompt is to determine whether a given word is semantically and economically associated with a particular GICS sector.
In practical terms, this simulates whether the appearance of the word in financial news could plausibly signal relevance or impact for that sector.

The objective of this prompt is to elicit from the model a binary judgment—whether a given lexical item is semantically and contextually associated with a specific GICS sector in financial discourse.
This allows for a simplified mapping of linguistic signals to sectoral impact, reducing ambiguity compared to multi-sector scoring methods.

---

Brown, T. B., et al. (2020). Language Models are Few-Shot Learners. Advances in Neural Information Processing Systems, 33, 1877–1901.

Liu, P., et al. (2023). Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing. ACM Computing Surveys, 55(9).

Reynolds, L., & McDonell, K. (2021). Prompt Programming for Large Language Models: Beyond the Few-Shot Paradigm. arXiv preprint arXiv:2102.07350.

The model’s ability to perform this task relies on in-context learning (Brown et al., 2020), where the linguistic instructions within the prompt condition the model’s internal reasoning toward a specific classification objective.
By providing explicit instructions and schema constraints, the model can perform consistent binary judgments aligned with human reasoning about financial relevance.

---

The prompt explicitly limits the task to a binary decision (“Yes” or “No”), as this reduces the risk of overgeneralization and facilitates downstream quantitative analysis.
By specifying the sector as an input, the model’s reasoning remains focused, improving precision.
The inclusion of a concise justification promotes interpretability and aligns with explainable AI principles.

---

The JSON schema enforces standardized outputs across runs, enabling systematic comparison and quantitative aggregation of results.
Confidence scores allow uncertainty analysis, while the free-text justification supports qualitative interpretation and human validation.

---

This design reflects how sector-specific language functions in financial communication.
For instance, words like “oil” or “refinery” have strong lexical and economic ties to the Energy sector, whereas generic terms such as “company” or “profit” occur across multiple sectors and are thus not indicative of sector-specific information.

---

This prompt operationalizes a prompt-engineered binary classifier for sector-specific lexical association, leveraging the contextual knowledge embedded in large language models.



In [17]:
words = {'Energy': ['oil', 'gas', 'drilling', 'company', 'coal', 'refining', 'production', 'well', 'exploration', 'refined', 'integrated', 'transportation', 'rig', 'pipeline', 'consumable', 'fuel', 'mining', 'engaged', 'equipment', 'marketing', 'service', 'primarily', 'involved', 'product', 'storage', 'related', 'classified', 'contractor', 'evaluation', 'completion', 'midstream', 'slurry', 'shipping', 'contract', 'least', 'one', 'significant', 'independent', 'energy', 'trader', 'natural', 'generation', 'metallurgical', 'coking', 'steel', 'either', 'chemical', 'supply', 'producing', 'industrial'], 'Materials': ['chemical', 'mining', 'metal', 'gold', 'company', 'aluminum', 'classified', 'paper', 'steel', 'packaging', 'product', 'fertilizer', 'mineral', 'industrial', 'building', 'manufacturer', 'diversified', 'specialty', 'mine', 'bauxite', 'precious', 'primarily', 'gas', 'material', 'coal', 'including', 'produce', 'limited', 'agricultural', 'producer', 'related', 'plastic', 'paint', 'pigment', 'finished', 'container', 'iron', 'ore', 'copper', 'excludes', 'commodity', 'glass', 'construction', 'process', 'used', 'production', 'includes', 'basic', 'synthetic', 'fiber'], 'Industrials': ['service', 'company', 'equipment', 'machinery', 'providing', 'electrical', 'industrial', 'heavy', 'classified', 'support', 'includes', 'business', 'security', 'marine', 'consulting', 'passenger', 'manufacturer', 'excludes', 'component', 'construction', 'agricultural', 'transportation', 'defense', 'air', 'primarily', 'facility', 'research', 'good', 'commercial', 'office', 'related', 'aerospace', 'civil', 'engineering', 'wire', 'farm', 'pollution', 'control', 'printing', 'environmental', 'alarm', 'freight', 'logistics', 'airline', 'trucking', 'airport', 'railtracks', 'port', 'management', 'system'], 'Consumer Discretionary': ['store', 'manufacturer', 'operator', 'retail', 'home', 'classified', 'service', 'owner', 'accessory', 'consumer', 'includes', 'motorcycle', 'leisure', 'apparel', 'company', 'electronics', 'household', 'improvement', 'appliance', 'furnishing', 'automotive', 'excludes', 'product', 'part', 'computer', 'providing', 'specialty', 'auto', 'automobile', 'tire', 'rubber', 'tv', 'furniture', 'garden', 'luxury', 'footwear', 'general', 'equipment', 'related', 'sport', 'casino', 'gaming', 'educational', 'mainly', 'residential', 'including', 'personal', 'bicycle', 'housewares', 'toy'], 'Consumer Staples': ['food', 'product', 'packaged', 'producer', 'retail', 'drug', 'household', 'classified', 'owner', 'brewer', 'distiller', 'vintner', 'beverage', 'meat', 'tobacco', 'operator', 'store', 'hypermarket', 'super', 'package', 'paper', 'agricultural', 'including', 'market', 'personal', 'consumer', 'center', 'distributor', 'company', 'excludes', 'directly', 'staple', 'respectively', 'beer', 'malt', 'brewery', 'alcoholic', 'drink', 'milk', 'grower', 'plantation', 'dairy', 'fruit', 'juice', 'poultry', 'fish', 'pet', 'cigarette', 'detergent', 'soap'], 'Health Care': ['health', 'care', 'service', 'hospital', 'biotechnology', 'includes', 'company', 'supply', 'drug', 'research', 'development', 'providing', 'tool', 'medical', 'genetic', 'pharmaceutical', 'product', 'device', 'managed', 'primarily', 'equipment', 'support', 'application', 'instrument', 'facility', 'manufacturing', 'system', 'center', 'business', 'operator', 'technology', 'distributor', 'marketing', 'owner', 'production', 'classified', 'cardiovascular', 'orthopedic', 'diagnostic', 'eye', 'safety', 'needle', 'syringe', 'patient', 'dialysis', 'lab', 'clerical', 'collection', 'staffing', 'rehabilitation'], 'Financials': ['bank', 'financial', 'mortgage', 'banking', 'investment', 'insurance', 'institution', 'brokerage', 'company', 'lending', 'asset', 'finance', 'diversified', 'primarily', 'excludes', 'classified', 'regional', 'thrift', 'service', 'commercial', 'activity', 'specialized', 'corporate', 'exchange', 'custody', 'reinsurance', 'casualty', 'providing', 'security', 'health', 'diverse', 'whose', 'significant', 'range', 'life', 'property', 'business', 'management', 'capital', 'related', 'market', 'small', 'holding', 'financing', 'credit', 'fund', 'provider', 'consumer', 'including', 'derived'], 'Information Technology': ['electronic', 'semiconductor', 'equipment', 'software', 'manufacturer', 'service', 'component', 'technology', 'system', 'processing', 'data', 'company', 'communication', 'hardware', 'peripheral', 'computer', 'includes', 'storage', 'classified', 'information', 'producing', 'instrument', 'infrastructure', 'developing', 'database', 'phone', 'management', 'including', 'excludes', 'outsourced', 'cellular', 'solar', 'industry', 'distributor', 'consulting', 'internet', 'application', 'business', 'product', 'providing', 'integration', 'outsourcing', 'automation', 'cloud', 'web', 'hosting', 'designed', 'enterprise', 'lan', 'wan'], 'Communication Services': ['telecommunication', 'online', 'network', 'television', 'radio', 'service', 'entertainment', 'cable', 'wireless', 'broadcasting', 'company', 'interactive', 'gaming', 'internet', 'distribution', 'includes', 'primarily', 'advertising', 'satellite', 'content', 'platform', 'provider', 'offering', 'movie', 'medium', 'classified', 'also', 'producing', 'operator', 'providing', 'marketing', 'producer', 'including', 'alternative', 'carrier', 'access', 'end', 'user', 'public', 'relation', 'publishing', 'publisher', 'newspaper', 'magazine', 'print', 'format', 'screening', 'show', 'music', 'theater'], 'Utilities': ['utility', 'electricity', 'gas', 'company', 'renewable', 'water', 'energy', 'using', 'power', 'electric', 'source', 'oil', 'solar', 'distribute', 'hydropower', 'wind', 'excludes', 'exploration', 'independent', 'producer', 'transportation', 'distribution', 'produce', 'involved', 'system', 'production', 'storage', 'classified', 'nuclear', 'main', 'charter', 'transmit', 'addition', 'core', 'redistribute', 'ipps', 'specialist', 'biomass', 'geothermal', 'installers', 'photovoltaic', 'provision', 'also', 'whose', 'natural', 'refined', 'purchase', 'treatment', 'trader', 'trading'], 'Real Estate': ['real', 'estate', 'property', 'reit', 'trust', 'development', 'leasing', 'ownership', 'operation', 'acquisition', 'management', 'company', 'engaged', 'industrial', 'operating', 'mall', 'shopping', 'home', 'diversified', 'hotel', 'resort', 'office', 'residential', 'health', 'service', 'care', 'activity', 'includes', 'including', 'type', 'warehouse', 'assisted', 'living', 'multifamily', 'apartment', 'student', 'housing', 'outlet', 'neighborhood', 'community', 'income', 'spectrum', 'purpose', 'develop', 'sell', 'agent', 'appraiser', 'across', 'two', 'serving']}

In [18]:
model_name = "Qwen/Qwen2.5-7B-Instruct"                     # [315e0, 69797]
model_name = "google/gemma-7b-it"                           # [315e0]
model_name = "meta-llama/Llama-3.1-8B-Instruct"             # [315e0, 9d5c9, 69797]

# !pip install -q transformers accelerate torch huggingface_hub pandas

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from huggingface_hub import login
# import torch
# import time
# import re
# import json
# import pandas as pd
# import warnings
# warnings.filterwarnings("ignore")

# login(token=HF_TOKEN)  # from src.secrets (see src/secrets_example.py)

# # --- Load model and tokenizer ---
# model_name = 
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # Load model on GPU automatically
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype="auto",
# )

# # --- Define prompt ---
# def label_word_sector(word, sector, model, tokenizer):
#     prompt = f"""You are a senior financial analyst and language model evaluator.
# Your task is to assess whether the given word or short phrase is semantically and contextually associated 
# with the specified GICS sector, in the sense that its appearance in a financial news headline 
# would likely indicate or directly impact that sector.

# Follow these precise rules:
# - Interpret the word in its most common financial or economic context.
# - Evaluate only the given sector (do not compare with others).
# - Some words (e.g., "company", "market", "growth") are too generic to associate with any single sector — these should be marked as 'No association'.
# - Consider relevance in the context of financial reporting and investor sentiment.
# - Base your decision on general world knowledge of how the word relates to that sector’s activities, assets, or risks.
# - Respond ONLY with a JSON object matching the following schema and nothing else.
# - Do NOT include explanations, multiple examples, or any text outside the JSON object.
# - End your output immediately after the closing curly brace.

# JSON schema:
# {{
#   "word": "{word}",
#   "sector": "{sector}",
#   "is_associated": "<Yes|No>",
#   "confidence": <0-100>,
#   "reason": "<one-sentence justification, ≤20 words>"
# }}

# Now analyze the relationship between "{word}" and the "{sector}" sector and return only the JSON object."""

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=150,
#             temperature=0.0,
#             do_sample=False,
#             eos_token_id=tokenizer.eos_token_id,
#             repetition_penalty=1.0
#         )

#     text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     text_generated = text.split(prompt)[-1].strip()
#     match = re.search(r"\{.*?\}", text_generated, re.S)
#     if not match:
#         return {"word": word, "sector": sector, "error": "no JSON"}
#     try:
#         return json.loads(match.group(0))
#     except Exception:
#         return {"word": word, "sector": sector, "error": "invalid JSON"}

# # --- Results ---
# records = []
# for sector in words:
#     for word in words[sector]:
#         result = label_word_sector(word, sector, model, tokenizer)
#         records.append({
#             "sector": sector,
#             "word": word,
#             f"{model_name}": result
#         })
#         print(sector, word)
# df = pd.DataFrame(records)

# df.to_parquet("gics_word_labeling.parquet", engine="pyarrow", index=False)

In [19]:
# # Merge Kaggle results (one model per notebook for parallel execution)

# import pandas as pd
# from functools import reduce

# df00 = pd.read_parquet("news/gics_word_labeling_0.parquet")
# df01 = pd.read_parquet("news/gics_word_labeling_1.parquet")
# df02 = pd.read_parquet("news/gics_word_labeling_2.parquet")

# dfs = [df00, df01, df02]
# merged_df = reduce(lambda left, right: pd.merge(left, right, on=["sector", "word"], how="outer"), dfs)

# errors = 0
# for i, row in merged_df.iterrows():
#     llama = row["meta-llama/Llama-3.1-8B-Instruct"].get("error", None)
#     gemma = row["google/gemma-7b-it"].get("error", None)
#     qwen = row["Qwen/Qwen2.5-7B-Instruct"].get("error", None)

#     if llama or gemma or qwen:
#         #print(f"Row {i} errors:")
#         if llama:
#             #print(f"  meta-llama/Llama-3.1-8B-Instruct: {llama}")
#             errors += 1
#         if gemma:
#             #print(f"  google/gemma-7b-it: {gemma}")
#             errors += 1
#         if qwen:
#             #print(f"  Qwen/Qwen2.5-7B-Instruct: {qwen}")
#             errors += 1
# print(f"Total errors: {25}")

# a = pd.read_parquet("gics_word_labeling_3_labeled.parquet")
# # Some errors in Llama due to invalid JSON due to incomplete output - needed more max tokens

# for i, row in merged_df.iterrows():
#     llama = row["meta-llama/Llama-3.1-8B-Instruct"]
#     if llama.get("error", None) is not None:
#         word = row["word"]
#         sector = row["sector"]

#         for _, r in a.iterrows():
#             if r["word"] == word and r["sector"] == sector:
#                 merged_df.at[i, "meta-llama/Llama-3.1-8B-Instruct"] = r["meta-llama/Llama-3.1-8B-Instruct"]
#                 break

# #merged_df.to_parquet("news/gics_word_labeling_full.parquet", engine="pyarrow", index=False)

In [20]:
import pandas as pd

merged_df = pd.read_parquet("news/gics_word_labeling_full.parquet", engine="pyarrow")
merged_df.head()

,sector,word,meta-llama/Llama-3.1-8B-Instruct,google/gemma-7b-it,Qwen/Qwen2.5-7B-Instruct
0,Communication Services,access,"{'confidence': 80.0, 'error': None, 'is_associ...","{'confidence': 80, 'is_associated': 'Yes', 're...","{'confidence': 85, 'is_associated': 'Yes', 're..."
1,Communication Services,advertising,"{'confidence': 80.0, 'error': None, 'is_associ...","{'confidence': 80, 'is_associated': 'Yes', 're...","{'confidence': 95, 'is_associated': 'Yes', 're..."
2,Communication Services,also,"{'confidence': 0.0, 'error': None, 'is_associa...","{'confidence': 80, 'is_associated': 'Yes', 're...","{'confidence': 0, 'is_associated': 'No', 'reas..."
3,Communication Services,alternative,"{'confidence': 80.0, 'error': None, 'is_associ...","{'confidence': 80, 'is_associated': 'Yes', 're...","{'confidence': 10, 'is_associated': 'No', 'rea..."
4,Communication Services,broadcasting,"{'confidence': 80.0, 'error': None, 'is_associ...","{'confidence': 80, 'is_associated': 'Yes', 're...","{'confidence': 95, 'is_associated': 'Yes', 're..."


In [21]:
merged_df["is_associated_summary"] = None
for i, row in merged_df.iterrows():
    map = {"Yes": 1, "No": -1}
    
    llama_take = row["meta-llama/Llama-3.1-8B-Instruct"]["is_associated"]
    gemma_take = row["google/gemma-7b-it"]["is_associated"]
    qwen_take = row["Qwen/Qwen2.5-7B-Instruct"]["is_associated"]

    merged_df.at[i, "is_associated_summary"] = sum([
        map.get(llama_take, 0),
        map.get(gemma_take, 0),
        map.get(qwen_take, 0)
    ])
    
merged_df["is_associated_summary"].value_counts()


is_associated_summary
3     297
1     158
-1     68
-3     27
Name: count, dtype: int64

In [22]:
df = merged_df[["sector", "word", "is_associated_summary"]]
#df.to_excel("news/gics_word_labeling_summary.xlsx", index=False)

In [23]:
merged_df["association_type"] = merged_df["is_associated_summary"].apply(
    lambda x: "positive" if x > 0 else ("negative" if x < 0 else "neutral")
)

sector_counts = merged_df.groupby(["sector", "association_type"]).size().unstack(fill_value=0)

print(sector_counts)

association_type        negative  positive
sector                                    
Communication Services         8        42
Consumer Discretionary         9        41
Consumer Staples              10        40
Energy                        11        39
Financials                     8        42
Health Care                    6        44
Industrials                    8        42
Information Technology         8        42
Materials                      4        46
Real Estate                    8        42
Utilities                     15        35


In [24]:
%reset -f

# instead lets try generating the words based on the gics sectors sub industry descriptions

In [25]:
import pandas as pd

df = pd.read_excel("news/gics_description.xlsx", header=4)

In [26]:
# current_sector = None
# for i, row in df.iterrows():
#     if i % 2 == 0:
#         current_sector = row['Unnamed: 1'] if not pd.isna(row['Sector']) else current_sector

#         sector = current_sector
#         sub_sector = row['Unnamed: 7']
#         description = df.iloc[i+1, 7]

#         print(f"{sector}\t{sub_sector}\t{description}")

#     else:
#         pass

In [27]:
# model_name = "Qwen/Qwen2.5-7B-Instruct"                     # [315e0, 69797]
# #model_name = "google/gemma-7b-it"                           # [315e0]
# #model_name = "meta-llama/Llama-3.1-8B-Instruct"             # [315e0, 9d5c9, 69797]

# !pip install -q transformers accelerate torch huggingface_hub pandas

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from huggingface_hub import login
# import torch
# import time
# import re
# import json
# import pandas as pd
# import warnings
# warnings.filterwarnings("ignore")

# login(token=HF_TOKEN)  # from src.secrets (see src/secrets_example.py)

# # --- Load model and tokenizer ---
# model_name = "Qwen/Qwen2.5-7B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # Load model on GPU automatically
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype="auto",
# )

# # --- Define prompt ---
# def label_word_sector(sector, subind, description, model, tokenizer):
#     prompt = f"""You are a senior financial analyst and language model evaluator.
# Your task is to generate a concise set of keywords that best represent the economic activity
# of a given GICS sector, sub-industry, and description — optimized specifically for identifying
# relevant financial news headlines.

# Follow these precise rules:
# - Produce a list of **up to 10 words or short phrases** (not sentences).
# - Focus on terms that strongly signal the sub-industry in news coverage.
# - Choose words that indicate the sector’s activities, assets, risks, or business model.
# - Avoid generic financial terms (e.g., “company,” “market,” “economy,” “growth”).
# - Avoid overly narrow jargon unless it commonly appears in news headlines.
# - Avoid repeating words already present in the sector or sub-industry name.
# - Do not use punctuation except for commas to separate items.
# - Respond ONLY with a JSON object matching the schema below and nothing else.
# - End output immediately after the closing curly brace.

# JSON schema:
# {{
#   "sector": "<sector>",
#   "sub_industry": "<sub_industry>",
#   "keywords": ["<word1>", "<word2>", ...]
# }}

# Now analyze the following GICS information and generate the keywords:

# Sector: f"{sector}"
# Sub-Industry: "{subind}"
# Description: "{description}"
# """

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=300,
#             temperature=0.0,
#             do_sample=False,
#             eos_token_id=tokenizer.eos_token_id,
#             repetition_penalty=1.0
#         )

#     text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
#     #text_generated = text.split(prompt)[-1].strip()
#     #match = re.search(r"\{.*?\}", text_generated, re.S)
#     #match = re.search(r"```json\s*(\{.*?\})\s*```", text_generated, re.S)
#     #if not match:
#     #    print("no match")
#     #    return {"sub_industry": subind, "sector": sector, "error": text_generated}
#     #try:
#     #    return json.loads(match.group(0))
#     #except Exception as e:
#     #    print(e)
#     #    return {"sub_industry": subind, "sector": sector, "error": text_generated}

#     marker = "GICS information and generate the keywords:"
#     start_pos = text.find(marker)
#     if start_pos == -1:
#         return {"sub_industry": subind, "sector": sector, "error": "Marker not found in text.", "output": text}
    
#     text_after = text[start_pos + len(marker):]
#     start = text_after.find('{')
#     end = text_after.find('}', start)
#     json_str = text_after[start:end+1]
#     try:
#         data = json.loads(json_str)
#         return data
#     except json.JSONDecodeError:
#         return {"sub_industry": subind, "sector": sector, "error": "Failed to parse JSON.", "output": text}

# # --- Results ---
# records = []
# for row in gics_subind_descriptions:
#     sector, subind, description = row.split("\t")
    
#     #result = label_word_sector(sector, subind, description, model, tokenizer)
#     #records.append({
#     #    "sector": sector,
#     #    "sub_industry": subind,
#     #    f"{model_name}": result
#     #})
#     #print(sector, subind, result)
#     print(sector, subind)
# df = pd.DataFrame(records)

# df.to_parquet("subind_words.parquet", engine="pyarrow", index=False)

notebookda71f0f990

In [28]:
# import pandas as pd
# mod1 = pd.read_parquet("news/subind_words.parquet")
# mod2 = pd.read_parquet("news/subind_words-2.parquet")
# mod3 = pd.read_parquet("news/subind_words-3.parquet")

# for row in mod1["meta-llama/Llama-3.1-8B-Instruct"]:
#     if (list(row.keys()) == ['keywords', 'sector', 'sub_industry']):
#         pass
#     else:
#         print(row)

# for row in mod2["google/gemma-7b-it"]:
#     if (list(row.keys()) == ['keywords', 'sector', 'sub_industry']):
#         pass
#     else:
#         print(row)

# for row in mod3["Qwen/Qwen2.5-7B-Instruct"]:
#     if (list(row.keys()) == ['keywords', 'sector', 'sub_industry']):
#         pass
#     else:
#         print(row)

# # merge on sector and sub_industry
# dfs = [mod1, mod2, mod3]
# merged_df = dfs[0]
# for df in dfs[1:]:
#     merged_df = pd.merge(merged_df, df, on=["sector", "sub_industry"], how="outer")

# merged_df.to_parquet("news/subind_words_full.parquet", engine="pyarrow", index=False)

In [29]:
import pandas as pd
import json

df = pd.read_parquet("news/subind_words_full.parquet", engine="pyarrow")
df

,sector,sub_industry,meta-llama/Llama-3.1-8B-Instruct,google/gemma-7b-it,Qwen/Qwen2.5-7B-Instruct
0,Communication Services,Advertising,"{'keywords': ['advertising', 'marketing', 'pub...","{'keywords': ['Advertising', 'Marketing', 'Pub...","{'keywords': ['advertising', 'marketing', 'pub..."
1,Communication Services,Alternative Carriers,"{'keywords': ['fiber-optic', 'cable', 'network...","{'keywords': ['High-bandwidth', 'Fiber-optic',...","{'keywords': ['fiber-optic', 'high-bandwidth',..."
2,Communication Services,Broadcasting,"{'keywords': ['television', 'radio', 'broadcas...","{'keywords': ['Television', 'Radio', 'Broadcas...","{'keywords': ['television', 'radio', 'broadcas..."
3,Communication Services,Cable & Satellite,"{'keywords': ['cable', 'satellite', 'televisio...","{'keywords': ['Cable TV', 'Satellite TV', 'Cab...","{'keywords': ['cable', 'satellite', 'televisio..."
4,Communication Services,Integrated Telecommunication Services,"{'keywords': ['telecom', 'networks', 'wireless...","{'keywords': ['Telecommunications', 'Fixed-lin...","{'keywords': ['telecom', 'networks', 'internet..."
...,...,...,...,...,...
153,Utilities,Gas Utilities,"{'keywords': ['gas', 'distribution', 'transmis...","{'keywords': ['Gas distribution', 'Gas transmi...","{'keywords': ['gas distribution', 'transmissio..."
154,Utilities,Independent Power Producers & Energy Traders,"{'keywords': ['power', 'energy', 'trading', 'm...","{'keywords': ['Independent Power Producers', '...","{'keywords': ['power', 'trading', 'generation'..."
155,Utilities,Multi-Utilities,"{'keywords': ['electricity', 'gas', 'water', '...","{'keywords': ['Utility', 'Multi-Utilities', 'E...","{'keywords': ['diversified operations', 'elect..."
156,Utilities,Renewable Electricity,"{'keywords': ['renewable', 'electricity', 'sol...","{'keywords': ['Renewable electricity', 'Electr...","{'keywords': ['renewable energy', 'electricity..."


In [30]:
words_per_gics_sector = {}
for i, row in df.iterrows():
    sector = row["sector"]
    keywords1 = row["Qwen/Qwen2.5-7B-Instruct"]["keywords"]
    keywords2 = row["google/gemma-7b-it"]["keywords"]
    keywords3 = row["meta-llama/Llama-3.1-8B-Instruct"]["keywords"]
    if sector not in words_per_gics_sector:
        words_per_gics_sector[sector] = []
    
    words_per_gics_sector[sector].extend(keywords1)
    words_per_gics_sector[sector].extend(keywords2)
    words_per_gics_sector[sector].extend(keywords3)

In [31]:
import spacy
nlp = spacy.load("en_core_web_sm")

def word2lemmas(word):
    word = word.lower()
    doc = nlp(word)
    lemmas = [token.lemma_ for token in doc]
    return tuple(lemmas)

### setence/word -> list of lemmas -> tuple of lemmas -> join tuple with " "

In [ ]:
# Count occurrences of each group of lemmas per sector
words_per_gics_sector_counted = {}
for sector, keywords in words_per_gics_sector.items():
    for word in keywords:
        lemmas = word2lemmas(word)
        words_per_gics_sector_counted[sector] = words_per_gics_sector_counted.get(sector, {})
        words_per_gics_sector_counted[sector][lemmas] = words_per_gics_sector_counted[sector].get(lemmas, 0) + 1

#words_per_gics_sector_counted
# {'Communication Services': {('advertise',): 5,
#   ('marketing',): 3,
#   ('public', 'relation'): 3,
#   ('data', '-', 'transmission'): 1,
#   ('communication',): 2,

# Count occurrences of the actual lemmas
keywords_gics = {}
for sector in words_per_gics_sector_counted:
    keywords_gics[sector] = {}

    # single words
    for tuple in words_per_gics_sector_counted[sector]:
        if len(tuple) == 1:
            word = tuple[0]
            keywords_gics[sector][word] = keywords_gics[sector].get(word, 0) + 1
    
    # multiple words
    for tuple in words_per_gics_sector_counted[sector]:
        if len(tuple) != 1:
            original = " ".join(tuple)

            for word in tuple:
                # if the signle word is relevant, count it
                if word in keywords_gics[sector]:
                    keywords_gics[sector][word] += 1

            keywords_gics[sector][original] = keywords_gics[sector].get(original, 0) + 1

#keywords_gics
# {'Communication Services': {'advertise': 1,
#   'marketing': 1,
#   'roam': 1,
#   'voice': 1,
#   'transmission - service': 1,
#   'alternative carrier': 1,


# Sort keywords by frequency per sector keepint top 
for sector, keyword_counts in keywords_gics.items():
    sorted_keywords = sorted(keyword_counts.items(), key=lambda x: x[1], reverse=True)[:30]
    keywords_gics[sector] = {kw: count for kw, count in sorted_keywords}
# with open("news/subind_words_count.json", "w") as f:
#     json.dump(keywords_gics, f, indent=2)
# print("Saved to news/subind_words_count.json")

Saved to news/subind_words_count.json
